In [66]:
import pandas as pd
import numpy as np

Obsługa **indeksowania hierarchicznego** jest ważnym elementem biblioteki pandas umożliwiającym
przypisanie do jednej osi wielu poziomów indeksowania (przypisanie dwóch lub więcej indeksów).

In [67]:
data = pd.Series(np.random.uniform(size=9),
                 index=[["a", "a", "a", "b", "b", "c", "c", "d", "d"],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])

In [68]:
data

a  1    0.122882
   2    0.338722
   3    0.128555
b  1    0.867206
   3    0.557958
c  1    0.216010
   2    0.701157
d  2    0.180057
   3    0.041250
dtype: float64

Wyświetloną czytelnie serią, której indeksem jest obiekt MultiIndex.

In [69]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

Obiekty o indeksie hierarchicznym obsługują tzw. **indeksowanie częściowe**. Indeksowanie to umożliwia zwięzły wybór podzbioru danych

In [70]:
data["b"]

,0
1,0.867206
3,0.557958


In [71]:
data["b":"c"]

b  1    0.867206
   3    0.557958
c  1    0.216010
   2    0.701157
dtype: float64

In [72]:
data.loc[["b", "d"]]

b  1    0.867206
   3    0.557958
d  2    0.180057
   3    0.041250
dtype: float64

## Przykład na danych

In [73]:
# Dane sprzedażowe w dwóch miastach, w dwóch latach
df = pd.DataFrame({
    "miasto":   ["Łódź", "Łódź", "Kraków", "Kraków"],
    "rok":      [2023,   2024,   2023,     2024],
    "sprzedaz": [120,    145,    200,      230]
})
df

,miasto,rok,sprzedaz
0,Łódź,2023,120
1,Łódź,2024,145
2,Kraków,2023,200
3,Kraków,2024,230


In [74]:
df.index

RangeIndex(start=0, stop=4, step=1)

### Zwykły filtrowanie

Za każdym razem: filtr po mieście, filtr po roku, wybór kolumny. Trzy operacje dla jednej wartości.

In [75]:
# Sprzedaż w Łodzi w 2024 - działa, ale rozwlekle:
df[(df.miasto == "Łódź") & (df.rok == 2024)]["sprzedaz"]

,sprzedaz
1,145


In [76]:
# Sprzedaż w Krakowie w 2023 - znowu to samo:
df[(df.miasto == "Kraków") & (df.rok == 2023)]["sprzedaz"]

,sprzedaz
2,200


### MultiIndex - indeks jako "ścieżka"

In [77]:
# Ten sam zbiór, ale z hierarchicznym indeksem:
sprzedaz = pd.Series(
    [120, 145, 200, 230],
    index=[["Łódź", "Łódź", "Kraków", "Kraków"],
           [2023,   2024,   2023,     2024]]
)
print(sprzedaz)

Łódź    2023    120
        2024    145
Kraków  2023    200
        2024    230
dtype: int64


In [78]:
# Sam indeks to obiekt MultiIndex:
sprzedaz.index

MultiIndex([(  'Łódź', 2023),
            (  'Łódź', 2024),
            ('Kraków', 2023),
            ('Kraków', 2024)],
           )

### Indeksowanie - podstawowe wzorce

In [79]:
# Pojedyncza wartość - krotka (miasto, rok):
sprzedaz["Łódź", 2024]

np.int64(145)

In [80]:
# Wszystko dla Łodzi (wybór zewnętrznego poziomu):
sprzedaz["Łódź"]

,0
2023,120
2024,145


In [81]:
# Wszystkie miasta, ale tylko rok 2023 (wewnętrzny poziom):
sprzedaz.loc[:, 2023]

,0
Łódź,120
Kraków,200


In [82]:
# Zakres miast (uwaga: wymaga posortowanego indeksu):
sprzedaz.sort_index().loc["Kraków":"Łódź"]

Kraków  2023    200
        2024    230
Łódź    2023    120
        2024    145
dtype: int64

## Z Series do DataFrame i z powrotem

In [83]:
print(sprzedaz)

Łódź    2023    120
        2024    145
Kraków  2023    200
        2024    230
dtype: int64


In [84]:
# unstack - "wypchnięcie" wewnętrznego poziomu do kolumn:
tabela = sprzedaz.unstack()
print(tabela)
print(type(tabela))

        2023  2024
Kraków   200   230
Łódź     120   145
<class 'pandas.core.frame.DataFrame'>


In [85]:
# stack - operacja odwrotna: kolumny wracają do indeksu:
tab_stack = tabela.stack()
print(tab_stack)
print(type(tab_stack))

Kraków  2023    200
        2024    230
Łódź    2023    120
        2024    145
dtype: int64
<class 'pandas.core.series.Series'>


In [86]:
print(df)

   miasto   rok  sprzedaz
0    Łódź  2023       120
1    Łódź  2024       145
2  Kraków  2023       200
3  Kraków  2024       230


In [87]:
# Konwersja płaskiej ramki na hierarchiczną - set_index:
df_hier = df.set_index(["miasto", "rok"])
print(df_hier)
print(df_hier.index)

             sprzedaz
miasto rok           
Łódź   2023       120
       2024       145
Kraków 2023       200
       2024       230
MultiIndex([(  'Łódź', 2023),
            (  'Łódź', 2024),
            ('Kraków', 2023),
            ('Kraków', 2024)],
           names=['miasto', 'rok'])


In [88]:
# I z powrotem - reset_index:
df2 = df_hier.reset_index()
print(df2)

   miasto   rok  sprzedaz
0    Łódź  2023       120
1    Łódź  2024       145
2  Kraków  2023       200
3  Kraków  2024       230


### Tabele przestawne (ang. _Pivot Table_)

In [89]:
# Przekształcamy płaską ramkę w tabelę przestawną
df2_my_pivot = df.set_index(["miasto", "rok"])["sprzedaz"].unstack()
print(df2_my_pivot)

rok     2023  2024
miasto            
Kraków   200   230
Łódź     120   145


In [90]:
df_pivot = pd.pivot_table(df,
                         index="miasto",
                         columns="rok",
                         values="sprzedaz")
print(df_pivot)

rok      2023   2024
miasto              
Kraków  200.0  230.0
Łódź    120.0  145.0


In [91]:
df_kwartaly = pd.DataFrame({
    "miasto":   ["Łódź"]*4 + ["Kraków"]*4,
    "rok":      [2023]*4 + [2023]*4,
    "kwartal":  ["Q1","Q2","Q3","Q4"] * 2,
    "sprzedaz": [30, 25, 35, 30, 50, 45, 55, 50]
})

df_kwartaly_sum = pd.pivot_table(df_kwartaly,
                                index="miasto",
                                columns="rok",
                                values="sprzedaz",
                                aggfunc="sum")
print(df_kwartaly_sum)

rok     2023
miasto      
Kraków   200
Łódź     120


### Agregacje po poziomie

In [92]:
sprzedaz.index.names = ["miasto", "rok"]
sprzedaz.groupby(level="miasto").sum()

,0
miasto,
Kraków,430
Łódź,265


In [93]:
sprzedaz.groupby(level="rok").mean()

,0
rok,
2023,160.0
2024,187.5


### Agregacja w DataFrame z MultiIndex

In [94]:
frame = pd.DataFrame({
    "sprzedaz": [120, 145, 200, 230],
    "koszty":   [80,  90,  130, 140]
}, index=[["Łódź","Łódź","Kraków","Kraków"],
          [2023,  2024,  2023,    2024]])
frame.index.names = ["miasto", "rok"]
frame.groupby(level="miasto").sum()

,sprzedaz,koszty
miasto,,
Kraków,430,270
Łódź,265,170


In [95]:
frame.groupby(level="miasto").agg(["sum", "mean"])

sprzedaz        koszty       
            sum   mean    sum   mean
miasto                              
Kraków      430  215.0    270  135.0
Łódź        265  132.5    170   85.0

In [96]:
frame["sprzedaz"].groupby(level="miasto").agg(
    total="sum", srednia="mean", rozstep=lambda x: x.max()-x.min()
    )

,total,srednia,rozstep
miasto,,,
Kraków,430,215.0,30
Łódź,265,132.5,25


# *Zadania*

Sieć "ModaŁódź" ma sklepy w trzech miastach. Każdy sklep sprzedaje trzy kategorie produktów. Dane obejmują 4 kwartały 2023 i 2024 roku.

## **Zadanie 1 — Budowanie MultiIndex**

a) Utwórz ramkę `df_hi` z hierarchicznym indeksem (`miasto`, `kategoria`, `rok`, `kwartal`). Posortuj indeks.

b) Wyświetl liczbę poziomów indeksu i ich nazwy.

c) Ile unikalnych kombinacji indeksu istnieje? Użyj `.index` do odpowiedzi.

In [97]:
df = pd.read_csv("sklepy_moda.csv", index_col=0)

# a) MultiIndex z czterema poziomami
df_hi = df.set_index(["miasto", "kategoria", "rok", "kwartal"]).sort_index()
display(df_hi.head(8))

# b) Liczba poziomów i ich nazwy
print("Liczba poziomów:", df_hi.index.nlevels)
print("Nazwy poziomów:", df_hi.index.names)

# c) Liczba unikalnych kombinacji
print("Unikalnych kombinacji:", len(df_hi.index))


przychod_tys  koszt_tys  sztuki
miasto kategoria rok  kwartal                                 
Kraków Damska    2023 Q1               71.3       51.9     806
                      Q2               90.8       52.1    1179
                      Q3               77.4       54.5     886
                      Q4              116.9       76.5    1285
                 2024 Q1               73.0       44.7     843
                      Q2               89.2       65.3     869
                      Q3               89.0       53.0     759
                      Q4              115.3       70.1    1052

Liczba poziomów: 4
Nazwy poziomów: ['miasto', 'kategoria', 'rok', 'kwartal']
Unikalnych kombinacji: 72


## **Zadanie 2 — Selekcje na MultiIndex**

a) Wybierz wszystkie dane dla Łodzi.
    
b) Wybierz dane dla Łodzi, kategorii Damska.
    
c) Wybierz dane dla wszystkich miast, ale tylko rok 2024. (Wskazówka: użyj metodę `xs()`)

d) Wybierz Q4 z obu lat, ale tylko dla Krakowa i kategorii Męska.

In [98]:
# a) Wszystkie dane dla Łodzi
print("=== 2a — Łódź ===")
display(df_hi.loc["Łódź"])

# b) Łódź, kategoria Damska
print("\n=== 2b — Łódź, Damska ===")
display(df_hi.loc[("Łódź", "Damska")])

# c) Wszystkie miasta, rok 2024 — xs() wybiera po dowolnym poziomie
print("\n=== 2c — rok 2024 ===")
display(df_hi.xs(2024, level="rok"))

# d) Kraków, Męska, tylko kwartał Q4
print("\n=== 2d — Kraków, Męska, Q4 ===")
display(df_hi.xs(("Kraków", "Męska"), level=("miasto", "kategoria")).xs("Q4", level="kwartal"))


=== 2a — Łódź ===


przychod_tys  koszt_tys  sztuki
kategoria rok  kwartal                                 
Damska    2023 Q1               50.5       35.2     615
               Q2               59.3       34.5     539
               Q3               61.9       34.3     915
               Q4               81.8       58.6     775
          2024 Q1               49.5       30.2     577
               Q2               62.5       39.8     627
               Q3               53.3       32.4     563
               Q4               85.8       55.0    1157
Dziecięca 2023 Q1               20.6       14.7     216
               Q2               31.4       19.0     370
               Q3               33.1       18.7     493
               Q4               43.6       30.7     409
          2024 Q1               35.4       24.9     301
               Q2               42.5       26.4     374
               Q3               35.7       22.0     301
               Q4               54.0       33.1     554
Męska     2023 Q1               40.3       26.9     335
               Q2               42.9       28.8     394
               Q3               42.0       28.8     465
               Q4               63.5       36.5     728
          2024 Q1               47.3       29.0     550
               Q2               47.9       31.6     445
               Q3               52.3       29.7     490
               Q4               71.1       39.7     730


=== 2b — Łódź, Damska ===


przychod_tys  koszt_tys  sztuki
rok  kwartal                                 
2023 Q1               50.5       35.2     615
     Q2               59.3       34.5     539
     Q3               61.9       34.3     915
     Q4               81.8       58.6     775
2024 Q1               49.5       30.2     577
     Q2               62.5       39.8     627
     Q3               53.3       32.4     563
     Q4               85.8       55.0    1157


=== 2c — rok 2024 ===


przychod_tys  koszt_tys  sztuki
miasto  kategoria kwartal                                 
Kraków  Damska    Q1               73.0       44.7     843
                  Q2               89.2       65.3     869
                  Q3               89.0       53.0     759
                  Q4              115.3       70.1    1052
        Dziecięca Q1               48.1       28.7     629
                  Q2               56.7       35.4     704
                  Q3               50.6       28.7     700
                  Q4               79.7       48.9     741
        Męska     Q1               60.5       38.3     578
                  Q2               74.4       42.7     771
                  Q3               67.2       43.9     868
                  Q4              100.3       62.5    1484
Wrocław Damska    Q1               58.3       42.8     824
                  Q2               76.1       45.8     960
                  Q3               71.1       46.6     689
                  Q4              107.6       61.2    1536
        Dziecięca Q1               36.4       24.2     416
                  Q2               52.6       31.0     686
                  Q3               35.5       26.2     521
                  Q4               61.4       45.0     650
        Męska     Q1               53.3       29.4     486
                  Q2               61.2       40.4     786
                  Q3               52.4       36.3     506
                  Q4               87.5       53.8    1157
Łódź    Damska    Q1               49.5       30.2     577
                  Q2               62.5       39.8     627
                  Q3               53.3       32.4     563
                  Q4               85.8       55.0    1157
        Dziecięca Q1               35.4       24.9     301
                  Q2               42.5       26.4     374
                  Q3               35.7       22.0     301
                  Q4               54.0       33.1     554
        Męska     Q1               47.3       29.0     550
                  Q2               47.9       31.6     445
                  Q3               52.3       29.7     490
                  Q4               71.1       39.7     730


=== 2d — Kraków, Męska, Q4 ===


,przychod_tys,koszt_tys,sztuki
rok,,,
2023,89.4,63.8,1253
2024,100.3,62.5,1484


## **Zadanie 3 — pivot_table: roczne podsumowanie**

a) Utwórz tabelę przestawną `tab_miasta`, która pokaże **sumę przychodów** w wierszach per miasto, w kolumnach per rok.

b) Dodaj do tabeli kolumnę `zmiana_proc` — procentową zmianę przychodu między 2023 a 2024. Które miasto rosło najszybciej?

c) Utwórz drugą tabelę przestawną, w której wiersze to (miasto, kategoria), kolumny to rok, wartości to *średni przychód kwartalny*. Która kombinacja (miasto, kategoria) ma najwyższy średni przychód w 2024?

In [99]:
# a) Suma przychodów: wiersze = miasto, kolumny = rok
tab_miasta = pd.pivot_table(
    df, index="miasto", columns="rok",
    values="przychod_tys", aggfunc="sum"
)
display(tab_miasta)

# b) Procentowa zmiana między 2023 a 2024
tab_miasta["zmiana_proc"] = (
    (tab_miasta[2024] - tab_miasta[2023]) / tab_miasta[2023] * 100
)
display(tab_miasta)
print("Najszybszy wzrost:", tab_miasta["zmiana_proc"].idxmax())

# c) Średni przychód kwartalny: wiersze = (miasto, kategoria), kolumny = rok
tab_mk = pd.pivot_table(
    df, index=["miasto", "kategoria"], columns="rok",
    values="przychod_tys", aggfunc="mean"
)
display(tab_mk)
print("Najwyższy średni przychód w 2024:", tab_mk[2024].idxmax())


rok,2023,2024
miasto,,
Kraków,824.3,904.0
Wrocław,697.7,753.4
Łódź,570.9,637.3


rok,2023,2024,zmiana_proc
miasto,,,
Kraków,824.3,904.0,9.668810
Wrocław,697.7,753.4,7.983374
Łódź,570.9,637.3,11.630758


Najszybszy wzrost: Łódź


rok                  2023    2024
miasto  kategoria                
Kraków  Damska     89.100  91.625
        Dziecięca  46.700  58.775
        Męska      70.275  75.600
Wrocław Damska     70.775  78.275
        Dziecięca  41.600  46.475
        Męska      62.050  63.600
Łódź    Damska     63.375  62.775
        Dziecięca  32.175  41.900
        Męska      47.175  54.650

Najwyższy średni przychód w 2024: ('Kraków', 'Damska')


## **Zadanie 4 — Agregacja po poziomach**
a) Na `df_hi` oblicz sumę przychodów per miasto (agreguj po poziomie `"miasto"`).

b) Oblicz **średni przychód kwartalny per kategoria** (agreguj po poziomie `"kategoria"`).

c) Użyj `.agg(["sum", "mean", "max"])` na kolumnie `przychod_tys` pogrupowanej po `(miasto, rok)`. Który wiersz ma najwyższą wartość `max`?

In [100]:
# a) Suma przychodów per miasto
print("=== 4a — suma per miasto ===")
display(df_hi["przychod_tys"].groupby(level="miasto").sum())

# b) Średni przychód kwartalny per kategoria
print("\n=== 4b — średnia per kategoria ===")
display(df_hi["przychod_tys"].groupby(level="kategoria").mean())

# c) sum / mean / max pogrupowane po (miasto, rok)
print("\n=== 4c — agg po (miasto, rok) ===")
agg = df_hi["przychod_tys"].groupby(level=["miasto", "rok"]).agg(["sum", "mean", "max"])
display(agg)
print("Wiersz z najwyższym max:", agg["max"].idxmax())


=== 4a — suma per miasto ===


,przychod_tys
miasto,
Kraków,1728.3
Wrocław,1451.1
Łódź,1208.2



=== 4b — średnia per kategoria ===


,przychod_tys
kategoria,
Damska,75.987500
Dziecięca,44.604167
Męska,62.225000



=== 4c — agg po (miasto, rok) ===


sum       mean    max
miasto  rok                          
Kraków  2023  824.3  68.691667  116.9
        2024  904.0  75.333333  115.3
Wrocław 2023  697.7  58.141667   96.0
        2024  753.4  62.783333  107.6
Łódź    2023  570.9  47.575000   81.8
        2024  637.3  53.108333   85.8

Wiersz z najwyższym max: ('Kraków', np.int64(2023))


## **Zadanie 5 — Marża i ranking**
a) W `df_hi` dodaj kolumnę `marza_tys = przychod_tys − koszt_tys`.

b) Utwórz tabelę przestawną: *wiersze = miasto*, *kolumny = kategoria*, *wartości = suma marży*. Które miasto jest najbardziej zyskowne? Która kategoria generuje najwyższą marżę?

In [101]:
# a) Kolumna marzy
df_hi["marza_tys"] = df_hi["przychod_tys"] - df_hi["koszt_tys"]
display(df_hi["marza_tys"].head(8))

# b) Tabela przestawna sumy marży: wiersze = miasto, kolumny = kategoria
tab_marza = pd.pivot_table(
    df_hi.reset_index(), index="miasto", columns="kategoria",
    values="marza_tys", aggfunc="sum"
)
display(tab_marza)
print("Najbardziej zyskowne miasto:", tab_marza.sum(axis=1).idxmax())
print("Kategoria z najwyższą marżą:", tab_marza.sum(axis=0).idxmax())


miasto  kategoria  rok   kwartal
Kraków  Damska     2023  Q1         19.4
                         Q2         38.7
                         Q3         22.9
                         Q4         40.4
                   2024  Q1         28.3
                         Q2         23.9
                         Q3         36.0
                         Q4         45.2
Name: marza_tys, dtype: float64

kategoria,Damska,Dziecięca,Męska
miasto,,,
Kraków,254.8,155.6,201.6
Wrocław,203.0,112.9,183.9
Łódź,184.6,106.8,156.3


Najbardziej zyskowne miasto: Kraków
Kategoria z najwyższą marżą: Damska
